# Moving Average vs Naive Forecast Demo

This notebook provides an empirical evaluation comparing a 3-point moving average smoothing technique against a naive last-value persistence forecasting baseline on synthetic noisy time series data.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0', 'loguru==0.7.3')

In [ ]:
import json
import os
import urllib.request
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from loguru import logger

logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-29c492-empirical-audit-of-moving-average-baseli/main/round-1/experiment-1/demo/mini_demo_data.json"

def load_data():
    try:
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        print(f"Could not load from GitHub URL ({e}), falling back to local file...")
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

data_json = load_data()
print("Loaded dataset structure keys:", list(data_json.keys()))

## Configuration
Define tunable experiment parameters.

In [ ]:
NUM_TRIALS = 10
SERIES_LENGTH = 20
NOISE_STD = 1.0

## Generate Synthetic Noisy Series and Run Evaluation

In [ ]:
def generate_noisy_series(length: int = 20, noise_std: float = 1.0, seed: int = 42) -> np.ndarray:
    np.random.seed(seed)
    true_mean = 10.0
    series = true_mean + np.random.normal(0, noise_std, size=length)
    return series

logger.info(f"Starting evaluation with {NUM_TRIALS} trials, length={SERIES_LENGTH}, noise_std={NOISE_STD}")

examples = []
ma_errors = []
naive_errors = []

for i in range(NUM_TRIALS):
    seed_val = 1000 + i
    series = generate_noisy_series(length=SERIES_LENGTH, noise_std=NOISE_STD, seed=seed_val)
    true_next = 10.0 + np.random.normal(0, noise_std, size=None)
    
    ma_pred = float(np.mean(series[-3:]))
    naive_pred = float(series[-1])
    
    ma_err = (ma_pred - true_next) ** 2
    naive_err = (naive_pred - true_next) ** 2
    
    ma_errors.append(ma_err)
    naive_errors.append(naive_err)
    
    example = {
        "input": f"Synthetic time series of length {SERIES_LENGTH} with noise std {NOISE_STD}, seed {seed_val}",
        "output": f"True next value: {true_next:.4f}",
        "metadata_fold": i % 5,
        "predict_moving_average": f"{ma_pred:.4f}",
        "predict_naive": f"{naive_pred:.4f}",
        "metadata_mse_ma": float(ma_err),
        "metadata_mse_naive": float(naive_err)
    }
    examples.append(example)

mse_ma = float(np.mean(ma_errors))
mse_naive = float(np.mean(naive_errors))
improvement = float((mse_naive - mse_ma) / mse_naive * 100.0)

logger.info(f"Results -> MSE Moving Average: {mse_ma:.4f}, MSE Naive: {mse_naive:.4f}, Improvement: {improvement:.2f}%")

## Results & Visualization

In [ ]:
print("=== Summary Metrics ===")
print(f"Moving Average MSE: {mse_ma:.4f}")
print(f"Naive Forecast MSE: {mse_naive:.4f}")
print(f"MSE Reduction: {improvement:.2f}%")

plt.figure(figsize=(8, 5))
methods = ['Moving Average (3-pt)', 'Naive (Last-Value)']
mses = [mse_ma, mse_naive]
plt.bar(methods, mses, color=['skyblue', 'salmon'], width=0.5)
plt.ylabel('Mean Squared Error (MSE)')
plt.title('Comparison of Forecasting Methods on Noisy Time Series')
for i, v in enumerate(mses):
    plt.text(i, v + 0.02, f"{v:.4f}", ha='center', fontweight='bold')
plt.ylim(0, max(mses) * 1.2)
plt.show()